# 1. Import

In [1]:
import os
from glob import glob
import numpy as np
from scipy import stats
import pandas as pd
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# 2. Configuration

In [2]:
class Configuration:
    
    size = 384 # default: 384
    source_dir = f'../series_npy/{size}'
    save_dir = f'../series_npy/{size}-aggregated'
    continued = False

CFG = Configuration()

# 3. Function

In [3]:
def aggregate_npy(npy_paths):
    '''
    Aggregate .npy files by calculating mean, std, and kurtosis.
    '''
    images = [np.load(path) for path in npy_paths]
    images = np.array(images)
    
    mean_image = np.mean(images, axis=0).astype(np.uint8)
    midpoint_image = images[len(images) // 2].astype(np.uint8)
    std_image = np.std(images, axis=0).astype(np.uint8)
    kurtosis_image = stats.kurtosis(images, axis=0).astype(np.uint8)
    
    images_dict = dict(
        mean=mean_image,
        midpoint=midpoint_image,
        std=std_image,
        kurtosis=kurtosis_image
    )
    
    return images_dict

# 4. Convert

In [4]:
# Get SeriesInstanceUIDs
series_ids = os.listdir(f'../series_npy/{CFG.size}')

# Save aggregated .npy files
for series_id in tqdm(series_ids):
    # Make save directory
    os.makedirs(CFG.save_dir + f'/{series_id}', exist_ok=True)
    
    # Load .npy files
    npy_paths = glob(CFG.source_dir + f'/{series_id}/*.npy')
    npy_paths = [path.replace('\\', '/') for path in npy_paths]
    npy_paths = sorted(npy_paths)
    
    # Aggregate .npy files
    images_dict = aggregate_npy(npy_paths)
    
    # Save aggregated .npy files
    for key, image in images_dict.items():
        save_path = f'{CFG.save_dir}/{series_id}/{key}.npy'
        np.save(save_path, image)

  0%|          | 0/4348 [00:00<?, ?it/s]